# Fig 6B / S13B — Per-GWAS comparison of SuSiE-rRSS and SuSiE-RSS (data processing)

- **Fig 6B:** regions with a credible set under SuSiE-RSS and SuSiE-rRSS. Partition positive final SuSiE-rRSS regions by unchanged or changed exported CS membership; the SER fallback bar counts the positive final SuSiE-rRSS regions that used the fallback (a region without a credible set needs none), shown apart because fallback and a change in CS membership can overlap.
- **Fig S13B:** changed regions with 0, 1 or multiple CSs before and after adjustment, including regions that lose all reported CSs.

**Outputs:** `figure_data/Fig6B.AD_GWAS_Method_Compare.rds`, `figure_data/FigS13B.AD_GWAS_Changed_Regions_CS_Number.rds` (+ `.csv`)


In [1]:
source("../../R/utils.R")

eb_mix <- read_gwas_tables("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/")
eb_mix <- eb_mix %>% mutate(method = "eb_mix") %>% filter(cs_coverage_0.95 != "susie_rss_0")

susie_rss <- read_gwas_tables("//hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_baseline_r4_L10//export/")
susie_rss <- susie_rss %>% mutate(method = "susie_rss") %>% filter(cs_coverage_0.95 != "susie_rss_0")

result_full <- bind_rows(eb_mix, susie_rss)

SER_flag_tbl <- read_tsv("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/SER_flag.tsv")

Rows: 338 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): study, block_id, rds_file
lgl (1): serFallbackUsed

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [2]:
library(dplyr)

cs_per_region <- eb_mix %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("", "0")) %>%
  distinct(study, block_id, cs_coverage_0.95) %>%
  count(study, block_id, name = "n_cs")

# single vs multi
cs_per_region %>%
  mutate(n_effect = if_else(n_cs == 1, "1", ">1")) %>%
  count(n_effect)

# per study, if you want the breakdown across the 12 GWAS
cs_per_region %>%
  mutate(n_effect = if_else(n_cs == 1, "1", ">1")) %>%
  count(study, n_effect) %>%
  tidyr::pivot_wider(names_from = n_effect, values_from = n, values_fill = 0)

# full distribution (1, 2, 3, 4, ...)
count(cs_per_region, n_cs)

n_effect,n
<chr>,<int>
1,282
>1,56


study,1,>1
<chr>,<int>,<int>
AD_Bellenguez_2022,37,9
AD_Bellenguez_EADB_2022,12,1
AD_Bellenguez_EADI_2022,1,0
AD_Bellenguez_GRACE_2022,3,0
AD_Bellenguez_main_2026,62,13
AD_Bellenguez_nobiobank_2026,41,8
AD_Bellenguez_noproxy_2026,50,13
AD_Jansen_2021,10,2
AD_Kunkle_Stage1_2019,10,0


n_cs,n
<int>,<int>
1,282
2,48
3,7
4,1


In [3]:
n_distinct(cs_per_region$block_id)

[1] 108

In [4]:
## regions / CS effects per study and method (for the text)
study_summary_wide <- result_full %>%
  filter(cs_coverage_0.95 != 0) %>%
  group_by(study, method) %>%
  summarise(
    n_regions    = n_distinct(block_id),
    n_cs_effects = n_distinct(paste(block_id, cs_coverage_0.95)),
    .groups = "drop"
  ) %>%
  pivot_wider(
    names_from = method,
    values_from = c(n_regions, n_cs_effects),
    names_glue = "{method}_{.value}",
    values_fill = 0
  )

study_summary_wide %>% mutate(effect_diff = susie_rss_n_cs_effects - eb_mix_n_cs_effects)

study,eb_mix_n_regions,susie_rss_n_regions,eb_mix_n_cs_effects,susie_rss_n_cs_effects,effect_diff
<chr>,<int>,<int>,<int>,<int>,<int>
AD_Bellenguez_2022,46,47,57,63,6
AD_Bellenguez_EADB_2022,13,14,14,16,2
AD_Bellenguez_EADI_2022,1,1,1,1,0
AD_Bellenguez_GRACE_2022,3,3,3,3,0
AD_Bellenguez_main_2026,75,77,90,104,14
AD_Bellenguez_nobiobank_2026,49,50,59,63,4
AD_Bellenguez_noproxy_2026,63,65,77,93,16
AD_Jansen_2021,12,14,14,24,10
AD_Kunkle_Stage1_2019,10,10,10,19,9


In [5]:
## export sanity summary per method
export_summary <- function(df, source = "ebmix", pip_thresh = 0.025,
                           study_col = "study", region_col = "block_id",
                           cs_index_col = "cs_coverage_0.95", pip_col = "PIP") {
  in_cs <- df[[cs_index_col]] != 0                       # variant is in a credible set
  tibble(
    source                         = source,
    studies                        = n_distinct(df[[study_col]]),
    `exported rows`                = nrow(df),
    `min PIP`                      = min(df[[pip_col]], na.rm = TRUE),
    `rows below thr`               = sum(df[[pip_col]] < pip_thresh, na.rm = TRUE),
    `CS variants`                  = sum(in_cs, na.rm = TRUE),
    `CS variants, PIP >= thr`      = sum(in_cs & df[[pip_col]] >= pip_thresh, na.rm = TRUE),
    CSs                            = df %>% filter(.data[[cs_index_col]] != 0) %>%
                                       distinct(.data[[study_col]], .data[[region_col]], .data[[cs_index_col]]) %>% nrow(),
    `study-blocks`                 = n_distinct(paste(df[[study_col]], df[[region_col]])),
    `unique blocks`                = n_distinct(df[[region_col]])
  ) %>%
    rename(!!paste0("rows below ", pip_thresh)      := `rows below thr`,
           !!paste0("CS variants, PIP >= ", pip_thresh) := `CS variants, PIP >= thr`)
}

result_full %>%
  filter(method %in% c("eb_mix", "susie_rss")) %>%
  group_split(method) %>%
  map_dfr(~ export_summary(.x, source = unique(.x$method)))

source,studies,exported rows,min PIP,rows below 0.025,CS variants,"CS variants, PIP >= 0.025",CSs,study-blocks,unique blocks
<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>
eb_mix,12,2463,0.02503011,0,2463,2463,403,338,108
susie_rss,12,2726,0.02501180,0,2726,2726,505,352,109


In [6]:
## ---- SER fallback regions per study ----
single_count <- SER_flag_tbl %>%
  distinct(study, block_id, .keep_all = TRUE) %>%
  group_by(study) %>%
  summarize(
    single = sum(serFallbackUsed %in% TRUE),
    count = n()
  )

# ---- per block x study x method: n CSs + index-free CS signature ----
eff_count <- result_full %>%
  mutate(cs = as.character(cs_coverage_0.95)) %>%
  filter(!is.na(cs), cs != "0") %>%
  # one signature per credible set = its sorted variant IDs (index dropped)
  group_by(block_id, study, method, cs) %>%
  summarise(cs_sig = paste(sort(unique(variant_ID)), collapse = ","),
            .groups = "drop") %>%
  # region signature = sorted set of CS signatures
  group_by(block_id, study, method) %>%
  summarise(n_eff = n(),
            sig   = paste(sort(cs_sig), collapse = "|"),
            .groups = "drop")

# ---- regions with a CS per study per method ----
region_n <- eff_count %>%
  count(study, method, name = "n_region") %>%
  pivot_wider(names_from = method, values_from = n_region, values_fill = 0) %>%
  rename(susie_rss_n = susie_rss, rrss_n = eb_mix)

# ---- block-level: does SuSiE-rRSS differ from SuSiE-RSS? ----
block_cmp <- eff_count %>%
  pivot_wider(names_from = method, values_from = c(n_eff, sig),
              values_fill = list(sig = "", n_eff = 0L)) %>%
  mutate(different = coalesce(sig_eb_mix, "") != coalesce(sig_susie_rss, ""))

block_cmp <- block_cmp %>%
  left_join(SER_flag_tbl %>% distinct(study, block_id, serFallbackUsed),
            by = c("study", "block_id"))

In [7]:
# Count only regions retaining a final SuSiE-rRSS credible set in the stack.
stack_rrss <- block_cmp %>%
  filter(n_eff_eb_mix > 0) %>%
  mutate(cat = if_else(different, "Different", "Same as SuSiE-RSS")) %>%
  count(study, cat, name = "n_region") %>%
  complete(study = region_n$study,
           cat = c("Same as SuSiE-RSS", "Different"),
           fill = list(n_region = 0L)) %>%
  mutate(method = "SuSiE-rRSS")

bar_rss <- region_n %>%
  transmute(study, method = "SuSiE-RSS", cat = "SuSiE-RSS", n_region = susie_rss_n)

# SER fallback among the regions with a final SuSiE-rRSS credible set (a subset of the stack above).
# Fallback and a change in CS membership can overlap; use a separate bar.
ser_with_cs <- block_cmp %>%
  filter(n_eff_eb_mix > 0) %>%
  group_by(study) %>%
  summarise(single = sum(serFallbackUsed %in% TRUE), .groups = "drop")
bar_ser <- region_n %>%
  left_join(ser_with_cs, by = "study") %>%
  transmute(study, method = "SER fallback", cat = "SER fallback",
            n_region = coalesce(single, 0L))
## flagged regions per study: all analyzed (single_count) vs. with a final credible set (the bar)
left_join(single_count, ser_with_cs, by = "study", suffix = c("_all", "_with_cs"))

study_order <- region_n %>%
  arrange(desc(rrss_n), desc(susie_rss_n)) %>%
  mutate(lab = recode(study, !!!cohort_labels)) %>% pull(lab)

plot_df <- bind_rows(bar_rss, stack_rrss, bar_ser) %>%
  mutate(study = factor(recode(study, !!!cohort_labels), levels = study_order),
         method = factor(method, levels = c("SuSiE-RSS", "SuSiE-rRSS", "SER fallback")),
         cat = factor(cat, levels = names(pal_region_cat)),
         xnum = as.integer(study) + recode(as.character(method),
                   `SuSiE-RSS` = -0.27, `SuSiE-rRSS` = 0, `SER fallback` = 0.27))

check <- stack_rrss %>% group_by(study) %>% summarise(n = sum(n_region), .groups = "drop") %>%
  left_join(select(region_n, study, rrss_n), by = "study")
stopifnot(all(check$n == check$rrss_n), all(plot_df$n_region >= 0),
          all(bar_ser$n_region <= region_n$rrss_n[match(bar_ser$study, region_n$study)]))
save_data(plot_df, "Fig6B.AD_GWAS_Method_Compare")


study,single_all,count,single_with_cs
<chr>,<int>,<int>,<int>
AD_Bellenguez_2022,1,46,1
AD_Bellenguez_EADB_2022,2,13,2
AD_Bellenguez_EADI_2022,0,1,0
AD_Bellenguez_GRACE_2022,0,3,0
AD_Bellenguez_main_2026,1,75,1
AD_Bellenguez_nobiobank_2026,1,49,1
AD_Bellenguez_noproxy_2026,1,63,1
AD_Jansen_2021,0,12,0
AD_Kunkle_Stage1_2019,1,10,1


## Fig S12B: changed regions by number of credible sets

In [8]:
cs_bin <- function(n) factor(case_when(n == 0 ~ "0 CS", n == 1 ~ "1 CS", TRUE ~ "Multiple CS"),
                             levels = names(pal_cs_number))
changed <- block_cmp %>% filter(different)

## pooled over GWAS (for the text): rows = SuSiE-RSS, columns = SuSiE-rRSS
changed %>%
  count(`SuSiE-RSS` = cs_bin(n_eff_susie_rss), `SuSiE-rRSS` = cs_bin(n_eff_eb_mix)) %>%
  pivot_wider(names_from = `SuSiE-rRSS`, values_from = n, values_fill = 0L, names_sort = TRUE)

## per GWAS, before vs after
changed_cs <- changed %>%
  transmute(study, `SuSiE-RSS` = n_eff_susie_rss, `SuSiE-rRSS` = n_eff_eb_mix) %>%
  pivot_longer(-study, names_to = "method", values_to = "n_cs") %>%
  count(study, method, cs = cs_bin(n_cs), name = "n_region") %>%
  complete(study, method, cs, fill = list(n_region = 0L)) %>%
  mutate(study  = factor(recode(study, !!!cohort_labels), levels = study_order),   # same order as Fig 6C
         method = factor(method, levels = c("SuSiE-RSS", "SuSiE-rRSS")))

changed_cs %>% group_by(method, cs) %>% summarise(n_region = sum(n_region), .groups = "drop")

save_data(changed_cs, "FigS13B.AD_GWAS_Changed_Regions_CS_Number")

SuSiE-RSS,0 CS,1 CS,Multiple CS
<fct>,<int>,<int>,<int>
1 CS,10,12,0
Multiple CS,4,22,20


method,cs,n_region
<fct>,<fct>,<int>
SuSiE-RSS,0 CS,0
SuSiE-RSS,1 CS,22
SuSiE-RSS,Multiple CS,46
SuSiE-rRSS,0 CS,14
SuSiE-rRSS,1 CS,34
SuSiE-rRSS,Multiple CS,20


In [9]:
# Supplementary multiple-CS frequencies, from the same region counts as above.
multiple_cs_frequency <- cs_per_region %>%
  group_by(study) %>%
  summarise(n_single = sum(n_cs == 1), n_multiple = sum(n_cs > 1),
            n_positive = n(), .groups = "drop") %>%
  mutate(multiple_percent = 100 * n_multiple / n_positive)
save_data(multiple_cs_frequency, "Fig6E.AD_GWAS_Multiple_CS_Frequency")
